# Foundry IQ knowledge base with Fabric Data Agent integration

This notebook creates a Foundry IQ knowledge base that combines indexed company documents with a **Fabric Data Agent Knowledge Source**.

- **Fabric Data Agent Knowledge Source**: connects Azure AI Search to a published data agent in a Fabric workspace for structured and analytical questions.
- **Delegated token**: uses the signed-in user's identity when Azure AI Search queries the protected Fabric source.

The Fabric Data Agent knowledge source is currently exposed by the preview Azure AI Search SDK even though its public documentation is limited.

## Step 1: Set up Azure AI Search, Azure OpenAI, and Fabric configuration

Load the resource configuration from the project `.env` file. This notebook requires the ID of the published Fabric Data Agent created during provisioning.

In [ ]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
FABRIC_TENANT_ID = os.environ["FABRIC_TENANT_ID"]
FABRIC_WORKSPACE_ID = os.environ["FABRIC_WORKSPACE_ID"]
FABRIC_DATA_AGENT_ID = os.environ["FABRIC_DATA_AGENT_ID"]

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"
HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
FABRIC_DATA_AGENT_KNOWLEDGE_SOURCE_NAME = "fabric-data-agent-knowledge-source"
KNOWLEDGE_BASE_NAME = "multisource-fabric-data-agent-knowledge-base"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
print("Environment variables loaded")

## Step 2: Authenticate with a delegated user token

The retrieval call needs an OAuth token for a signed-in user who can access the Fabric workspace and published data agent. Azure AI Search forwards this delegated identity when querying the protected source.

Run the cell below to sign in and request the Azure AI Search scope used by `query_source_authorization`.

In [ ]:
from azure.identity import InteractiveBrowserCredential

user_credential = InteractiveBrowserCredential(tenant_id=FABRIC_TENANT_ID)
user_token = user_credential.get_token("https://search.azure.com/.default").token

print("Acquired token for logged-in user")

## Step 3: Create the search index knowledge sources

Create or update the same HR and health document sources used by the other multi-source labs.

- `hrdocs-knowledge-source`: points to the `hrdocs` search index
- `healthdocs-knowledge-source`: points to the `healthdocs` search index

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

index_client = SearchIndexClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    credential=search_credential,
    api_version=AZURE_SEARCH_API_VERSION,
)

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "Contoso HR documents"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "Contoso health benefits documents"),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="uid"),
                SearchIndexFieldReference(name="snippet_parent_id"),
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="snippet"),
            ],
            search_fields=[SearchIndexFieldReference(name="snippet")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)

print("Search index knowledge sources created")

## Step 4: Create the Fabric Data Agent knowledge source

The preview `azure-search-documents` package exposes a Fabric Data Agent source that identifies the published agent by its Fabric workspace ID and data agent ID.

Because this API is not yet in the stable SDK, first verify that the notebook kernel has a preview package containing the required public model classes.

In [ ]:
from azure.search.documents.indexes import models as search_index_models
from azure.search.documents.knowledgebases import models as retrieval_models

required_models = {
    "azure.search.documents.indexes.models": (
        search_index_models,
        ["FabricDataAgentKnowledgeSource", "FabricDataAgentKnowledgeSourceParameters"],
    ),
    "azure.search.documents.knowledgebases.models": (
        retrieval_models,
        ["FabricDataAgentKnowledgeSourceParams"],
    ),
}
missing_models = [
    f"{module_name}.{model_name}"
    for module_name, (module, model_names) in required_models.items()
    for model_name in model_names
    if not hasattr(module, model_name)
]
if missing_models:
    raise RuntimeError(
        "The installed azure-search-documents package does not support Fabric Data Agent knowledge sources. "
        f"Missing models: {', '.join(missing_models)}. Install the version pinned in notebooks/requirements.txt."
    )

print("Fabric Data Agent knowledge source models are available")

In [ ]:
from azure.search.documents.indexes.models import (
    FabricDataAgentKnowledgeSource,
    FabricDataAgentKnowledgeSourceParameters,
)

fabric_data_agent_knowledge_source = FabricDataAgentKnowledgeSource(
    name=FABRIC_DATA_AGENT_KNOWLEDGE_SOURCE_NAME,
    description="Published Contoso Fabric Data Agent for structured product and inventory analysis",
    fabric_data_agent_parameters=FabricDataAgentKnowledgeSourceParameters(
        workspace_id=FABRIC_WORKSPACE_ID,
        data_agent_id=FABRIC_DATA_AGENT_ID,
    ),
)
index_client.create_or_update_knowledge_source(
    knowledge_source=fabric_data_agent_knowledge_source
)

print("Fabric Data Agent knowledge source created")

## Step 5: Create the multi-source knowledge base

Combine the two document indexes and the Fabric Data Agent with an Azure OpenAI chat model. Answer synthesis lets the knowledge base produce one response across the selected sources, while low reasoning effort enables source selection and query planning.

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

knowledge_source_names = [
    HR_KNOWLEDGE_SOURCE_NAME,
    HEALTH_KNOWLEDGE_SOURCE_NAME,
    FABRIC_DATA_AGENT_KNOWLEDGE_SOURCE_NAME,
 ]

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Multi-source knowledge base combining company documents with a Fabric Data Agent",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in knowledge_source_names],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_instructions=(
        "Use the Fabric Data Agent for structured product, inventory, and analytical questions. "
        "Use the search indexes for HR and health policy documents."
    ),
)

index_client.create_or_update_knowledge_base(knowledge_base)
print("Knowledge base created")

## Step 6: Query the knowledge base

Ask a question that requires current inventory analysis from the Fabric Data Agent and role information from the indexed HR documents. The data agent is always queried for this request; agentic retrieval can decide whether each document source is also needed.

Retrieval can take longer than an index-only query because Azure AI Search invokes the published Fabric Data Agent.

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    FabricDataAgentKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    SearchIndexKnowledgeSourceParams,
)
from IPython.display import Markdown, display

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    knowledge_base_name=KNOWLEDGE_BASE_NAME,
    credential=search_credential,
    api_version=AZURE_SEARCH_API_VERSION,
)

question = (
    "I'm a Contoso buyer preparing for the summer sale. "
    "Which products or product categories have the lowest or zero inventory right now, and where? "
    "Which Contoso job roles are responsible for inventory strategy and budget oversight?"
)

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HR_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HEALTH_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        FabricDataAgentKnowledgeSourceParams(
            knowledge_source_name=FABRIC_DATA_AGENT_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=True,
        ),
    ],
    include_activity=True,
    max_runtime_in_seconds=180,
)

result = knowledge_base_client.retrieve(
    retrieval_request=retrieval_request,
    query_source_authorization=user_token,
)
display(Markdown(result.response[0].content[0].text))

## Step 7: Inspect retrieval activity

Review the query plan and source operations. A successful mixed-source request should include a Fabric Data Agent activity record and any search-index queries selected by agentic retrieval.

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

## Step 8: Inspect search and Data Agent references

Print each reference as returned by the preview API. Compare the document citations with the Fabric Data Agent reference, synthesized response, and structured source data.

Do not assume the Data Agent payload uses the same fields as a `fabricOntology` reference; inspect its actual `type` and `sourceData` values.

In [ ]:
references_json = [reference.as_dict() for reference in result.references or []]
print(json.dumps(references_json, indent=2))